# Benchmark Robust Asymmetric Loss (RAL) — XGB / LightGBM / CatBoost / torch-MLP, CXR vs CXR+EHR

RAL (Park et al. ICCVW'23, chuyên CXR long-tailed). 1 hàm `ral_loss` torch dùng chung:
- **torch-MLP**: train trực tiếp (đa nhãn 13).
- **XGB/LightGBM/CatBoost**: custom objective — `torch.autograd` tính grad+hess (clip hess>0).
- 13 nhãn CheXpert, LP-FT fullmicro, subject 70/10/20, default + tuned-F2, heatmap + Δ. Output file MỚI.

## 0. Cài CatBoost (nếu chưa có)

In [ ]:
!pip -q install catboost

## 1. Load fusion + merge nhãn thô (giống PLAN E)

In [ ]:
import os, numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)
WRITE_DIR='/content/drive/MyDrive/lpft_out'
SRC_DIR  ='/content/drive/MyDrive/MIMIC_CXR/planB_out'

df = pd.read_parquet(os.path.join(WRITE_DIR,'fusion_dataset_lpft_fullmicro.parquet'))
CHEX13=['Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion',
 'Pleural Other','Pneumonia','Pneumothorax']
cm = pd.read_parquet(os.path.join(SRC_DIR,'manifest_cohort.parquet'))
cmr= cm[['dicom_id']+CHEX13].drop_duplicates('dicom_id').rename(columns={c:'raw_'+c for c in CHEX13})
df = df.merge(cmr, on='dicom_id', how='left')
RAW=['raw_'+c for c in CHEX13]
print('df:', df.shape, '| raw merged:', all(c in df.columns for c in RAW))

## 2. Split subject 70/10/20 + preprocess

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
rng=np.random.default_rng(42); uniq=np.unique(df.subject_id.values); rng.shuffle(uniq)
n=len(uniq); s1,s2=int(.70*n),int(.80*n)
trs,vas,tes=set(uniq[:s1]),set(uniq[s1:s2]),set(uniq[s2:])
sp=np.where(df.subject_id.isin(trs),'train',np.where(df.subject_id.isin(vas),'val','test'))
tr=sp=='train'; va=sp=='val'; te=sp=='test'
assert not (trs&tes) and not (trs&vas) and not (vas&tes)
print('subject 70/10/20 (ảnh):', {k:int((sp==k).sum()) for k in ['train','val','test']})

IMG=[f'img_{i}' for i in range(768)]
DROP={'dicom_id','subject_id','study_id','hadm_id','stay_id','first_hadm_id','first_admittime',
      'first_icd10_codes','study_dt','admittime','ed_intime','encounter_time','split','view',
      'disease_group','chiefcomplaint'}
if df['gender'].dtype=='object': df['gender']=df['gender'].map({'M':1,'F':0})
EHR=[c for c in df.columns if not c.startswith(('img_','y_','dx_','raw_')) and c not in DROP]
FEATSETS={'CXR':IMG, 'CXR+EHR':IMG+EHR}
print('IMG:',len(IMG),'| EHR:',len(EHR))
def prep(cols):
    X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy('float32')
    X=SimpleImputer(strategy='median').fit(X[tr]).transform(X)
    return StandardScaler().fit(X[tr]).transform(X).astype('float32')
Xcache={k:prep(v) for k,v in FEATSETS.items()}
RAWm=df[RAW].to_numpy('float32')           # 1/0/-1/NaN theo dòng df
# nhãn train U-Ones
def uones(L): return np.where(np.isnan(L),0.,np.where(L==-1,1.,L)).astype('float32')
Yuo=uones(RAWm)
print({k:v.shape for k,v in Xcache.items()})

## 3. RAL loss + custom objective (autograd grad+hess) + train + eval
`ral_loss` torch dùng chung. Boosting: grad+hess qua autograd, clip hess>0.

In [ ]:
import torch, torch.nn as nn
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,fbeta_score,
                             average_precision_score,roc_auc_score)
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- RAL (Eq.6, log-ASL + Hill) ----
GP,GN,CLIP,LAM,EPS = 0.0, 4.0, 0.05, 1.5, 1e-8
def ral_loss(z, y):
    p=torch.sigmoid(z)
    Lp = y*(1-p).clamp(min=EPS).pow(GP)*(-torch.log(p.clamp(min=EPS)))
    pt = (p-CLIP).clamp(min=0)
    Ln = (LAM-p)*(1-y)*pt.pow(GN)*(-torch.log((1-pt).clamp(min=EPS)))
    return Lp+Ln                                   # per-element
def ral_grad_hess(margin, y):
    z=torch.tensor(np.asarray(margin,dtype=np.float64), requires_grad=True)
    yt=torch.tensor(np.asarray(y,dtype=np.float64))
    L=ral_loss(z,yt).sum()
    g,=torch.autograd.grad(L,z,create_graph=True); h,=torch.autograd.grad(g.sum(),z)
    return g.detach().numpy(), np.clip(h.detach().numpy(),1e-6,None)

# ---- objectives per library ----
def obj_xgb(y_true,y_pred): return ral_grad_hess(y_pred,y_true)
def obj_lgb(y_true,y_pred): return ral_grad_hess(y_pred,y_true)
class RALObjCat:
    def calc_ders_range(self, approxes, targets, weights):
        g,h=ral_grad_hess(np.array(approxes), np.array(targets)); d1=-g; d2=-h
        if weights is not None: w=np.array(weights); d1=d1*w; d2=d2*w
        return list(zip(d1.tolist(), d2.tolist()))
sig=lambda z: 1/(1+np.exp(-np.clip(z,-30,30)))

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SAMPLE_TRAIN=None
tr_idx=np.where(tr)[0]; va_idx=np.where(va)[0]; te_idx=np.where(te)[0]
if SAMPLE_TRAIN and len(tr_idx)>SAMPLE_TRAIN:
    tr_idx=np.random.default_rng(0).choice(tr_idx,SAMPLE_TRAIN,replace=False)

def macro_ap(S, R):            # S:(N,13) score, R:(N,13) raw; loại -1, blank->0
    aps=[]
    for j in range(13):
        col=R[:,j]; k=col!=-1; yt=np.where(np.isnan(col[k]),0.,col[k]).astype(int)
        if yt.sum()>0: aps.append(average_precision_score(yt,S[k,j]))
    return float(np.mean(aps))

# ---- train từng model -> trả score matrix val/test (N,13) ----
def train_boost(kind, X):
    Sv=np.zeros((len(va_idx),13)); St=np.zeros((len(te_idx),13))
    for j in tqdm(range(13), desc=kind, leave=False):
        ytr=Yuo[tr_idx,j]
        if kind=='XGBoost':
            m=XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.1,tree_method='hist',
                            objective=obj_xgb,n_jobs=-1).fit(X[tr_idx],ytr)
            mv=m.predict(X[va_idx],output_margin=True); mt=m.predict(X[te_idx],output_margin=True)
        elif kind=='LightGBM':
            m=LGBMClassifier(n_estimators=300,learning_rate=0.1,num_leaves=31,
                             objective=obj_lgb,n_jobs=-1,verbose=-1).fit(X[tr_idx],ytr)
            mv=m.predict(X[va_idx],raw_score=True); mt=m.predict(X[te_idx],raw_score=True)
        else: # CatBoost
            m=CatBoostClassifier(iterations=300,depth=4,learning_rate=0.1,loss_function=RALObjCat(),
                                 eval_metric='AUC',verbose=0).fit(X[tr_idx],ytr)
            mv=m.predict(X[va_idx],prediction_type='RawFormulaVal'); mt=m.predict(X[te_idx],prediction_type='RawFormulaVal')
        Sv[:,j]=sig(mv); St[:,j]=sig(mt)
    return Sv, St

def train_mlp(X):
    d=X.shape[1]
    net=nn.Sequential(nn.Linear(d,512),nn.ReLU(),nn.Dropout(0.3),
                      nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,13)).to(dev)
    opt=torch.optim.AdamW(net.parameters(),lr=1e-3,weight_decay=1e-2)
    Xt=torch.tensor(X); Yt=torch.tensor(Yuo)
    best=-1; best_state=None; bad=0
    for ep in range(40):
        net.train(); perm=np.random.default_rng(ep).permutation(tr_idx)
        for i in range(0,len(perm),4096):
            b=perm[i:i+4096]; xb=Xt[b].to(dev); yb=Yt[b].to(dev)
            opt.zero_grad(); loss=ral_loss(net(xb),yb).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): Sv=torch.sigmoid(net(Xt[va_idx].to(dev))).cpu().numpy()
        ap=macro_ap(Sv, RAWm[va_idx])
        if ap>best: best=ap; best_state={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=6: break
    net.load_state_dict(best_state); net.eval()
    with torch.no_grad():
        Sv=torch.sigmoid(net(Xt[va_idx].to(dev))).cpu().numpy(); St=torch.sigmoid(net(Xt[te_idx].to(dev))).cpu().numpy()
    return Sv, St

# ---- eval: default(0.5) + tuned(F2) per nhãn ----
GRID=np.linspace(0.02,0.98,49)
def best_thr(y,s):
    f2=[fbeta_score(y,(s>=t).astype(int),beta=2,zero_division=0) for t in GRID]; return float(GRID[int(np.argmax(f2))])
def metr(y,pred): return dict(accuracy=accuracy_score(y,pred),
    precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),
    f1=f1_score(y,pred,zero_division=0),f2=fbeta_score(y,pred,beta=2,zero_division=0))

MODELS=['XGBoost','LightGBM','CatBoost','MLP']
CKPT=os.path.join(WRITE_DIR,'benchmark_ral_lpft.csv')
if os.path.exists(CKPT): os.remove(CKPT)
for fs in FEATSETS:
    X=Xcache[fs]
    for mname in MODELS:
        Sv,St = train_mlp(X) if mname=='MLP' else train_boost(mname,X)
        rows=[]
        for j,c in enumerate(CHEX13):
            rv=RAWm[va_idx,j]; kv=rv!=-1; yv=np.where(np.isnan(rv[kv]),0.,rv[kv]).astype(int); sv=Sv[kv,j]
            rt=RAWm[te_idx,j]; kt=rt!=-1; yt=np.where(np.isnan(rt[kt]),0.,rt[kt]).astype(int); st=St[kt,j]
            au=roc_auc_score(yt,st) if 0<yt.sum()<len(yt) else np.nan
            ap=average_precision_score(yt,st) if yt.sum()>0 else np.nan
            m_def=metr(yt,(st>=0.5).astype(int))
            tthr=best_thr(yv,sv); m_tun=metr(yt,(st>=tthr).astype(int))
            for mode,mm,thr in [('default',m_def,0.5),('tuned',m_tun,tthr)]:
                rows.append(dict(model=mname,featset=fs,label=c,thr_mode=mode,thr=round(float(thr),4),
                    **mm, auprc=ap, auroc=au, prevalence=round(float(yt.mean()),4), n_pos=int(yt.sum())))
        pd.DataFrame(rows).to_csv(CKPT, mode='a', header=not os.path.exists(CKPT), index=False)
        print(f'✓ {fs} {mname} | auprc(mean)={np.nanmean([r["auprc"] for r in rows]):.3f} | f2 tuned(mean)={np.mean([r["f2"] for r in rows if r["thr_mode"]=="tuned"]):.3f}')
res=pd.read_csv(CKPT); print('\nrows:', len(res), '(kỳ vọng 2×4×13×2 =', 2*4*13*2, ')')

## 4. Heatmap — per model (CXR vs CXR+EHR) + Δ (thêm EHR) 6 chỉ số

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
res=pd.read_csv(os.path.join(WRITE_DIR,'benchmark_ral_lpft.csv'))
METRICS=['accuracy','precision','recall','f1','f2','auprc']; MODELS_O=['XGBoost','LightGBM','CatBoost','MLP']
def hm(sub,ax,title):
    m=sub.set_index('label')[METRICS].reindex(CHEX13)
    sns.heatmap(m,ax=ax,cmap='RdYlGn',vmin=0,vmax=1,annot=True,fmt='.3f',linewidths=.5,linecolor='black',
                annot_kws={'size':7},cbar_kws={'label':'score'})
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(''); ax.set_yticklabels(ax.get_yticklabels(),rotation=0)
for mname in MODELS_O:
    fig,ax=plt.subplots(1,2,figsize=(16,6))
    for a,fs in zip(ax,['CXR','CXR+EHR']):
        hm(res[(res.model==mname)&(res.featset==fs)&(res.thr_mode=='tuned')],a,f'{mname} · {fs} · RAL tuned-F2')
    plt.tight_layout(); plt.show()
for metric in METRICS:
    D=pd.DataFrame(index=CHEX13)
    for mname in MODELS_O:
        a=res[(res.model==mname)&(res.featset=='CXR')     &(res.thr_mode=='tuned')].set_index('label')[metric].reindex(CHEX13)
        b=res[(res.model==mname)&(res.featset=='CXR+EHR')&(res.thr_mode=='tuned')].set_index('label')[metric].reindex(CHEX13)
        D[mname]=b-a
    plt.figure(figsize=(7,6))
    sns.heatmap(D,cmap='RdBu_r',center=0,vmin=-.15,vmax=.15,annot=True,fmt='+.3f',linewidths=.5,linecolor='black',
                annot_kws={'size':8},cbar_kws={'label':f'Δ {metric}'})
    plt.title(f'Δ (CXR+EHR − CXR) · {metric} · RAL tuned (đỏ=EHR giúp)'); plt.ylabel(''); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()
print('== MACRO (mean 13 nhãn) =='); print(res.groupby(['model','featset','thr_mode'])[METRICS].mean().round(4))